In [1]:
import pandas as pd

df = pd.read_csv('aice_day4_customer_churn.csv')

In [2]:
df.head()

,customer_id,age,gender,city,plan,tenure_months,monthly_fee,support_calls,late_payments,churn
0,1001,60.0,M,Seoul,Premium,36,61000.0,3,1,0
1,1002,26.0,M,Incheon,Basic,4,41000.0,3,1,0
2,1003,52.0,M,Busan,Premium,70,83000.0,3,3,0
3,1004,57.0,F,Seoul,Basic,55,73000.0,4,1,0
4,1005,NaN,F,Seoul,Basic,49,42000.0,5,2,0


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   customer_id    80 non-null     int64  
 1   age            76 non-null     float64
 2   gender         80 non-null     str    
 3   city           77 non-null     str    
 4   plan           80 non-null     str    
 5   tenure_months  80 non-null     int64  
 6   monthly_fee    77 non-null     float64
 7   support_calls  80 non-null     int64  
 8   late_payments  80 non-null     int64  
 9   churn          80 non-null     int64  
dtypes: float64(2), int64(5), str(3)
memory usage: 6.4 KB


In [ ]:
df.describe()

,customer_id,age,tenure_months,monthly_fee,support_calls,late_payments,churn
count,80.0000,76.000000,80.000000,77.000000,80.000000,80.000000,80.000000
mean,1040.5000,41.381579,37.200000,72168.831169,4.137500,2.400000,0.387500
std,23.2379,13.434252,19.940037,25738.769957,2.689613,1.514487,0.490253
min,1001.0000,20.000000,1.000000,30000.000000,0.000000,0.000000,0.000000
25%,1020.7500,28.750000,21.000000,54000.000000,1.750000,1.000000,0.000000
50%,1040.5000,41.500000,37.000000,68000.000000,4.000000,3.000000,0.000000
75%,1060.2500,54.000000,54.000000,89000.000000,6.000000,4.000000,1.000000
max,1080.0000,63.000000,71.000000,120000.000000,8.000000,5.000000,1.000000


In [6]:
# 결측치 처리
df['age'] = df['age'].fillna(df['age'].median())
df['city'] = df['city'].fillna(df['city'].mode().iloc[0])
df['monthly_fee'] = df['monthly_fee'].fillna(df['monthly_fee'].median())

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   customer_id    80 non-null     int64  
 1   age            80 non-null     float64
 2   gender         80 non-null     str    
 3   city           80 non-null     str    
 4   plan           80 non-null     str    
 5   tenure_months  80 non-null     int64  
 6   monthly_fee    80 non-null     float64
 7   support_calls  80 non-null     int64  
 8   late_payments  80 non-null     int64  
 9   churn          80 non-null     int64  
dtypes: float64(2), int64(5), str(3)
memory usage: 6.4 KB


In [8]:
df['gender'] = df['gender'].map({
    'M':0,
    'F':1
})

df = pd.get_dummies(df, columns=['city'])
df = pd.get_dummies(df, columns=['plan'])

In [9]:
df.head()

,customer_id,age,gender,tenure_months,monthly_fee,support_calls,late_payments,churn,city_Busan,city_Daegu,city_Incheon,city_Seoul,plan_Basic,plan_Premium,plan_Standard
0,1001,60.0,0,36,61000.0,3,1,0,False,False,False,True,False,True,False
1,1002,26.0,0,4,41000.0,3,1,0,False,False,True,False,True,False,False
2,1003,52.0,0,70,83000.0,3,3,0,True,False,False,False,False,True,False
3,1004,57.0,1,55,73000.0,4,1,0,False,False,False,True,True,False,False
4,1005,41.5,1,49,42000.0,5,2,0,False,False,False,True,True,False,False


In [10]:
df.isnull().sum()

customer_id      0
age              0
gender           0
tenure_months    0
monthly_fee      0
support_calls    0
late_payments    0
churn            0
city_Busan       0
city_Daegu       0
city_Incheon     0
city_Seoul       0
plan_Basic       0
plan_Premium     0
plan_Standard    0
dtype: int64

In [11]:
# x / y 분리
x = df.drop(columns=['customer_id','churn'])
y = df['churn']

In [13]:
x.shape

(80, 13)

In [14]:
y.shape

(80,)

In [22]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [23]:
print(x_train.shape ,x_test.shape, y_train.shape , y_test.shape)

(64, 13) (16, 13) (64,) (16,)


In [25]:
from sklearn.preprocessing import StandardScaler

scale = StandardScaler()

x_train_scaled = scale.fit_transform(x_train)
x_test_scaled = scale.transform(x_test)

In [27]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

model.fit(x_train_scaled, y_train)
pred = model.predict(x_test_scaled)

pred

array([1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0])

In [28]:
y_test.values

array([1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0])

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

acc = accuracy_score(y_test, pred) # 전체 문제 중 몇개 맞췄는지
pre = precision_score(y_test, pred) # 1이라고 예측한 것 중 실제 1 비율(FP)
recall = recall_score(y_test, pred) # 실제 1중 얼마나 찾아냈는가(FN)

print(acc, pre, recall)

0.75 0.6666666666666666 0.6666666666666666


In [ ]:
from sklearn.metrics import f1_score

f1 = f1_score(y_test, pred) # precision, recall 둘 다 잘 하고있는지
f1

0.6666666666666666

In [35]:
proba = model.predict_proba(x_test_scaled)
proba

array([[4.18688147e-01, 5.81311853e-01],
       [1.97189138e-01, 8.02810862e-01],
       [9.09522859e-01, 9.04771414e-02],
       [9.21148204e-01, 7.88517955e-02],
       [9.68492204e-01, 3.15077964e-02],
       [1.21525655e-01, 8.78474345e-01],
       [6.04042764e-01, 3.95957236e-01],
       [8.93740602e-01, 1.06259398e-01],
       [4.87655864e-01, 5.12344136e-01],
       [9.99694202e-01, 3.05798362e-04],
       [9.40953426e-01, 5.90465739e-02],
       [1.46913207e-01, 8.53086793e-01],
       [9.96190334e-01, 3.80966584e-03],
       [2.70647815e-01, 7.29352185e-01],
       [9.94580648e-01, 5.41935206e-03],
       [9.42422473e-01, 5.75775270e-02]])

In [38]:
proba_1 =proba[:,1]
proba_1

array([5.81311853e-01, 8.02810862e-01, 9.04771414e-02, 7.88517955e-02,
       3.15077964e-02, 8.78474345e-01, 3.95957236e-01, 1.06259398e-01,
       5.12344136e-01, 3.05798362e-04, 5.90465739e-02, 8.53086793e-01,
       3.80966584e-03, 7.29352185e-01, 5.41935206e-03, 5.75775270e-02])

In [39]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y_test, proba_1)
auc

0.7833333333333333

In [ ]:
## 임계값(threshold)를 0.3으로 조정
pred_03 = (proba_1 >= 0.3).astype(int)
print(pred, pred_03)

[1 1 0 0 0 1 0 0 1 0 0 1 0 1 0 0] [1 1 0 0 0 1 1 0 1 0 0 1 0 1 0 0]


In [41]:
# 기존 임계값 0.5
pred_pre = precision_score(y_test, pred)
pred_recall = recall_score(y_test, pred)

# 바꾼 임계값 0.3
pre_03 = precision_score(y_test, pred_03)
recall_03 = recall_score(y_test, pred_03)

print(pred_pre, pred_recall)
print(pre_03, recall_03)

0.6666666666666666 0.6666666666666666
0.5714285714285714 0.6666666666666666


In [43]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(max_depth=3, random_state=42)

tree_model.fit(x_train, y_train);

In [45]:
tree_pred = tree_model.predict(x_test)
tree_proba = tree_model.predict_proba(x_test)[:,1]

In [46]:
tree_acc = accuracy_score(y_test, tree_pred)
tree_precision = precision_score(y_test, tree_pred)
tree_recall = recall_score(y_test, tree_pred)
tree_f1 = f1_score(y_test, tree_pred)
tree_roc_auc = roc_auc_score(y_test, tree_proba)

print(tree_acc, tree_precision, tree_recall, tree_f1, tree_roc_auc)

0.875 0.75 1.0 0.8571428571428571 1.0


In [ ]:
log_acc = accuracy_score(y_test, pred)
log_precision = precision_score(y_test, pred)
log_recall = recall_score(y_test, pred)
log_f1 = f1_score(y_test, pred)
log_roc_auc = roc_auc_score(y_test, proba_1)

# LogisticRegression & DecisionTree 비교
print("LogisticRegression")
print(log_acc, log_precision, log_recall, log_f1, log_roc_auc)
X
print("DecisionTree")
print(tree_acc, tree_precision, tree_recall, tree_f1, tree_roc_auc)


LogisticRegression
0.75 0.6666666666666666 0.6666666666666666 0.6666666666666666 0.7833333333333333
DecisionTree
0.875 0.75 1.0 0.8571428571428571 1.0


In [48]:
for depth in [1, 2, 3, 4, 5, 10]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)

    tree.fit(x_train, y_train)

    train_pred = tree.predict(x_train)
    test_pred = tree.predict(x_test)

    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)

    print(depth, train_acc, test_acc)

1 0.78125 0.75
2 0.84375 1.0
3 0.921875 0.875
4 0.984375 0.8125
5 1.0 0.8125
10 1.0 0.8125


In [ ]:
from sklearn.model_selection import cross_val_score

for depth in [1, 2, 3, 4, 5, 10]:
    tree = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )

    scores = cross_val_score(
        tree,
        x_train,
        y_train,
        cv=5,
        scoring='accuracy'
    )

    # 교차검증 기준 
    print(depth, scores.mean())

1 0.7038461538461538
2 0.7474358974358974
3 0.7769230769230769
4 0.7166666666666667
5 0.7474358974358974
10 0.7320512820512821


In [50]:
final_model = DecisionTreeClassifier(max_depth=3, random_state=42)

final_model.fit(x_train, y_train)
final_pred = final_model.predict(x_test)
final_proba = final_model.predict_proba(x_test)[:,1]

final_acc = accuracy_score(y_test, final_pred)
final_prec = precision_score(y_test, final_pred)
final_recall = recall_score(y_test, final_pred)
final_f1 = f1_score(y_test, final_pred)

final_roc_auc = roc_auc_score(y_test, final_proba)

print(final_acc, final_prec, final_recall, final_f1, final_roc_auc)

0.875 0.75 1.0 0.8571428571428571 1.0
